<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/Android_Malware_Detection_Demo_ipynb_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Environment setup
import os
import sys
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Project paths
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Malicious project work"
)
BYTEPLOT_ROOT = PROJECT_ROOT / "Byteplot images"
MODEL_DIR = BYTEPLOT_ROOT / "model_outputs"
GENERATOR_PATH = PROJECT_ROOT / "byteplot_generator.py"

RESNET_CHECKPOINT = MODEL_DIR / "resnet50_best.pth"
MOBILENET_CHECKPOINT = MODEL_DIR / "mobilenet_v3_large_best.pth"

# Verify required files
required_files = [
    RESNET_CHECKPOINT,
    MOBILENET_CHECKPOINT,
    GENERATOR_PATH
]

for path in required_files:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

if not all(path.exists() for path in required_files):
    raise FileNotFoundError(
        "A required model checkpoint or generator file is missing."
    )

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("\nSelected device:", device)
print("Setup complete.")

Python: 3.13.16
PyTorch: 2.11.0+cpu
CUDA available: False
Mounted at /content/drive
OK: /content/drive/MyDrive/Malicious project work/Byteplot images/model_outputs/resnet50_best.pth
OK: /content/drive/MyDrive/Malicious project work/Byteplot images/model_outputs/mobilenet_v3_large_best.pth
OK: /content/drive/MyDrive/Malicious project work/byteplot_generator.py

Selected device: cpu
Setup complete.


In [2]:
# Cell 2: Load trained models

class_names = [
    "adware",
    "banking",
    "benign",
    "riskware",
    "sms"
]

def load_checkpoint(path):
    try:
        checkpoint = torch.load(
            path,
            map_location=device,
            weights_only=True
        )
    except TypeError:
        checkpoint = torch.load(path, map_location=device)

    if "model_state_dict" not in checkpoint:
        raise ValueError(f"Unexpected checkpoint format: {path}")

    saved_classes = checkpoint.get("class_names")
    if saved_classes is not None and list(saved_classes) != class_names:
        raise ValueError(
            f"Class order mismatch in {path.name}: {saved_classes}"
        )

    return checkpoint

# Re-create the same model architectures
resnet_model = models.resnet50(weights=None)
resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features, len(class_names)
)

mobile_model = models.mobilenet_v3_large(weights=None)
mobile_model.classifier[3] = nn.Linear(
    mobile_model.classifier[3].in_features, len(class_names)
)

# Load saved weights
resnet_checkpoint = load_checkpoint(RESNET_CHECKPOINT)
mobile_checkpoint = load_checkpoint(MOBILENET_CHECKPOINT)

resnet_model.load_state_dict(
    resnet_checkpoint["model_state_dict"]
)
mobile_model.load_state_dict(
    mobile_checkpoint["model_state_dict"]
)

resnet_model = resnet_model.to(device).eval()
mobile_model = mobile_model.to(device).eval()

print("ResNet50 loaded successfully")
print("Best validation macro-F1:",
      resnet_checkpoint.get("best_val_macro_f1"))

print("\nMobileNetV3-Large loaded successfully")
print("Best validation macro-F1:",
      mobile_checkpoint.get("best_val_macro_f1"))

print("\nClass order:", class_names)
print("Device:", device)

ResNet50 loaded successfully
Best validation macro-F1: 0.8972413447100939

MobileNetV3-Large loaded successfully
Best validation macro-F1: 0.8952395339807792

Class order: ['adware', 'banking', 'benign', 'riskware', 'sms']
Device: cpu


In [3]:
# Cell 3: Load the project's byteplot generator

generator_source = GENERATOR_PATH.read_text(encoding="utf-8")

exec(
    compile(generator_source, str(GENERATOR_PATH), "exec"),
    globals()
)

if "generate_precise_byteplot" not in globals():
    raise RuntimeError(
        "generate_precise_byteplot() was not found in the generator file."
    )

print("Byteplot generator loaded successfully.")
print("Function:", generate_precise_byteplot.__name__)
print("Target image size: 256 × 256")

Byteplot generator loaded successfully.
Function: generate_precise_byteplot
Target image size: 256 × 256


In [4]:
# Cell 4: Image preprocessing for model inference

IMAGE_SIZE = 224

preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Verify model output dimensions
dummy_input = torch.zeros(
    1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device
)

with torch.inference_mode():
    resnet_output = resnet_model(dummy_input)
    mobile_output = mobile_model(dummy_input)

assert resnet_output.shape == (1, len(class_names))
assert mobile_output.shape == (1, len(class_names))

print("Preprocessing configured.")
print("Input shape:", tuple(dummy_input.shape))
print("ResNet50 output shape:", tuple(resnet_output.shape))
print("MobileNetV3-Large output shape:", tuple(mobile_output.shape))
print("Class order:", class_names)
print("\nNote: normalization must still be verified against training.")

Preprocessing configured.
Input shape: (1, 3, 224, 224)
ResNet50 output shape: (1, 5)
MobileNetV3-Large output shape: (1, 5)
Class order: ['adware', 'banking', 'benign', 'riskware', 'sms']

Note: normalization must still be verified against training.


In [6]:
# Cell 5: APK validation and model inference

import os
import zipfile
import torch

MAX_APK_SIZE = 200 * 1024 * 1024
MAX_DEX_SIZE = 100 * 1024 * 1024


def analyze_apk(apk_path):
    if not apk_path or not os.path.isfile(apk_path):
        raise ValueError("Please provide a valid APK file.")

    if not apk_path.lower().endswith(".apk"):
        raise ValueError("The file must have an .apk extension.")

    apk_size = os.path.getsize(apk_path)
    if apk_size > MAX_APK_SIZE:
        raise ValueError("APK exceeds the 200 MB demo limit.")

    if not zipfile.is_zipfile(apk_path):
        raise ValueError("Invalid APK/ZIP archive.")

    with zipfile.ZipFile(apk_path, "r") as archive:
        bad_file = archive.testzip()
        if bad_file is not None:
            raise ValueError(f"Archive integrity check failed: {bad_file}")

        if "classes.dex" not in archive.namelist():
            raise ValueError("No classes.dex found in the APK.")

        dex_info = archive.getinfo("classes.dex")
        if dex_info.file_size > MAX_DEX_SIZE:
            raise ValueError("classes.dex exceeds the 100 MB demo limit.")

        dex_bytes = archive.read("classes.dex")

    if not dex_bytes:
        raise ValueError("classes.dex is empty.")

    byteplot = generate_precise_byteplot(
        dex_bytes, target_size=(256, 256)
    )
    image_tensor = preprocess(byteplot).unsqueeze(0).to(device)

    results = {}

    for model_name, model in {
        "ResNet50": resnet_model,
        "MobileNetV3-Large": mobile_model
    }.items():
        model.eval()

        with torch.inference_mode():
            logits = model(image_tensor)
            probabilities = torch.softmax(logits, dim=1)[0].cpu()

        predicted_index = int(probabilities.argmax())

        results[model_name] = {
            "predicted_class": class_names[predicted_index],
            "scores_percent": {
                name: round(float(prob) * 100, 2)
                for name, prob in zip(
                    class_names, probabilities.tolist()
                )
            }
        }

    report = {
        "filename": os.path.basename(apk_path),
        "apk_size_bytes": apk_size,
        "dex_size_bytes": len(dex_bytes),
        "analysis_status": "experimental",
        "model_results": results,
        "models_agree": (
            results["ResNet50"]["predicted_class"]
            == results["MobileNetV3-Large"]["predicted_class"]
        ),
        "lightgbm_status": (
            "Not run: required syscall/Binder features unavailable"
        ),
        "limitations": [
            "Byteplot generation has not been verified against training.",
            "Image normalization has not been verified against training.",
            "Predictions are not a confirmed malware verdict."
        ]
    }

    return report, byteplot


print("APK analysis function defined successfully.")
print("APK files will be inspected as archives, not executed.")

APK analysis function defined successfully.
APK files will be inspected as archives, not executed.


In [7]:
# Cell 6: Test with the existing APK

test_apk = "/content/com.iris.gallery_16.apk"

if os.path.isfile(test_apk):
    test_report, test_byteplot = analyze_apk(test_apk)

    print("APK:", test_report["filename"])
    print("Size:", f'{test_report["apk_size_bytes"]:,} bytes')
    print("DEX size:", f'{test_report["dex_size_bytes"]:,} bytes')
    print("\nModel agreement:", test_report["models_agree"])

    for model_name, result in test_report["model_results"].items():
        print(f"\n{model_name}: {result['predicted_class']}")
        print("Scores:", result["scores_percent"])

    print("\nStatus:", test_report["analysis_status"])
    display(test_byteplot)
else:
    print("Test APK not found at:", test_apk)
    print("Skip this test; the UI will accept an uploaded APK.")

Test APK not found at: /content/com.iris.gallery_16.apk
Skip this test; the UI will accept an uploaded APK.


In [8]:
# Cell 6: Gradio APK analysis interface

import gradio as gr
import json

def run_apk_analysis(apk_path):
    if not apk_path:
        return "Please upload an APK file.", None, None

    try:
        report, byteplot = analyze_apk(apk_path)

        model_results = report["model_results"]
        resnet = model_results["ResNet50"]
        mobile = model_results["MobileNetV3-Large"]

        markdown_report = f"""
# Android Malware Detection Report

**File:** `{report['filename']}`
**APK size:** {report['apk_size_bytes']:,} bytes
**DEX size:** {report['dex_size_bytes']:,} bytes

## Predictions

| Model | Predicted class | Model score |
|---|---|---:|
| ResNet50 | {resnet['predicted_class']} | {resnet['scores_percent'][resnet['predicted_class']]:.2f}% |
| MobileNetV3-Large | {mobile['predicted_class']} | {mobile['scores_percent'][mobile['predicted_class']]:.2f}% |

**Model agreement:** {"Yes" if report["models_agree"] else "No"}

## Class scores

| Class | ResNet50 | MobileNetV3-Large |
|---|---:|---:|
"""
        for label in class_names:
            markdown_report += (
                f"| {label} | {resnet['scores_percent'][label]:.2f}% "
                f"| {mobile['scores_percent'][label]:.2f}% |\n"
            )

        markdown_report += """
## Important limitations

- **Experimental results only:** this is not a confirmed malware verdict.
- The byteplot generation and image normalization have not been verified against the training pipeline.
- LightGBM was not run because the required syscall/Binder features are unavailable.
- The APK was inspected as an archive, not installed or executed.
"""
        return markdown_report, byteplot, report

    except Exception as exc:
        return (
            f"**Analysis failed:** `{type(exc).__name__}: {exc}`",
            None,
            None
        )


with gr.Blocks(title="Android Malware Detection") as demo:
    gr.Markdown("""
    # Android Malware Detection
    Upload an APK to obtain experimental byteplot-based predictions
    from ResNet50 and MobileNetV3-Large.
    """)

    apk_input = gr.File(
        label="Upload APK",
        file_types=[".apk"],
        type="filepath"
    )

    analyze_button = gr.Button("Analyze APK", variant="primary")

    report_output = gr.Markdown(label="Analysis report")

    with gr.Row():
        byteplot_output = gr.Image(label="Generated byteplot")
        json_output = gr.JSON(label="Detailed model results")

    analyze_button.click(
        fn=run_apk_analysis,
        inputs=apk_input,
        outputs=[report_output, byteplot_output, json_output]
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2884a0bff3c4a5a15f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
%pip -q install -U gradio plotly
print("Frontend dependencies ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 34.9 MB/s eta 0:00:00
Frontend dependencies ready.


In [ ]:
# Cell 7: MalwareGuard AI — Streamlined Interactive Security Dashboard with AI Chatbot

import os
import json
import gradio as gr
from google import genai
from google.colab import userdata

# Configure Gemini Client securely if key exists
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    GEMINI_AVAILABLE = True
except Exception:
    client = None
    GEMINI_AVAILABLE = False

CSS = """
.gradio-container {
    background: radial-gradient(circle at 10% 10%, #0d1117 0%, #161b22 100%) !important;
    color: #c9d1d9 !important;
    max-width: 1400px !important;
}
.glass {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 12px;
    padding: 20px;
    backdrop-filter: blur(12px);
    margin-bottom: 16px;
}
.hero {
    background: linear-gradient(135deg, rgba(31, 111, 235, 0.15) 0%, rgba(56, 139, 253, 0.1) 100%);
    border: 1px solid rgba(56, 139, 253, 0.2);
    border-radius: 16px;
    padding: 28px;
    margin-bottom: 20px;
}
.muted { color: #8b949e; }
"""

def generate_ai_report(report_dict):
    if not GEMINI_AVAILABLE:
        return "Gemini API key is not configured. Falling back to default system report.\n\n" + \
               "- The APK was analyzed using localized deep-learning models in the secure backend.\n" + \
               "- Final classification was aggregated across model weights.\n" + \
               "- Treat these screening results as experimental validation only."
    try:
        prompt = (
            f"You are MalwareGuard AI, an Android security specialist. "
            f"Analyze this technical report extracted from an APK and provide a professional breakdown: \n"
            f"{json.dumps(report_dict, indent=2)}\n"
        )
        response = client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=prompt,
        )
        return response.text
    except Exception as e:
        return f"Failed to query Gemini AI: {e}. Falling back to default analysis results."

def show_page(page):
    return (
        gr.update(visible=page == "Dashboard"),
        gr.update(visible=page == "Analyze APK"),
        f"**MalwareGuard AI** / {page}"
    )

with gr.Blocks(title="MalwareGuard AI") as app:

    gr.Markdown(
        "# ◈ MalwareGuard AI\n"
        "<span class='muted'>Next-generation Android security threat scanner and AI Assistant</span>"
    )
    page_label = gr.Markdown("**MalwareGuard AI** / Dashboard")

    with gr.Row():
        with gr.Column(scale=1, min_width=200):
            gr.Markdown("### Navigation")
            nav_dashboard = gr.Button("⌂  Dashboard", variant="primary")
            nav_analyze = gr.Button("⬆  Analyze APK")

            gr.Markdown(f"""
            <div class="glass">
              <b>ENGINE STATUS</b><br><br>
              Deep Learning · Ready<br>
              LightGBM · Loaded<br>
              Gemini AI · {"Active" if GEMINI_AVAILABLE else "Fallback mode"}<br>
              Security Engine · Online
            </div>
            """)

        with gr.Column(scale=4):

            # PAGE 1: DASHBOARD
            with gr.Column(visible=True) as dashboard_page:
                gr.HTML("""
                <div class="hero">
                  <h1>Intelligent Malware Prediction</h1>
                  <p class="muted">
                    Upload Android applications to scan binaries, identify behavioral patterns, and query Gemini security intelligence instantly.
                  </p>
                </div>
                """)

                gr.Markdown("### Quick Actions")
                dashboard_scan = gr.Button("＋  Scan New APK File", variant="primary")

                gr.Markdown("""
                <div class="glass">
                  <b>Secure Processing Pipeline</b><br><br>
                  APKs are parsed as isolated files in the secure container sandbox. Code footprints, structural artifacts, and model ensembles are calculated purely in the background to provide a unified risk classification.
                </div>
                """)

            # PAGE 2: APK ANALYSIS
            with gr.Column(visible=False) as analyze_page:
                gr.Markdown("## APK Security Scan")
                gr.Markdown("Select a local `.apk` file below to execute the background scanning and AI report pipeline.")

                with gr.Row():
                    with gr.Column(scale=1):
                        apk_input = gr.File(
                            label="Upload Target APK File",
                            file_types=[".apk"],
                            type="filepath"
                        )
                        analyze_btn = gr.Button("Execute Threat Scan", variant="primary")
                        report_download = gr.DownloadButton("Download JSON Report", visible=False)

                    with gr.Column(scale=2):
                        predicted_class_output = gr.HTML(label="Threat Assessment")

                # AI Report & Interactive Chatbot
                gr.Markdown("### AI Generated Threat Report")
                scan_report = gr.Markdown(value="Your analysis results and Gemini AI report will appear here...")

                gr.Markdown("### Talk to MalwareGuard Chatbot")
                chatbot = gr.Chatbot(label="AI Security Companion")
                chat_input = gr.Textbox(placeholder="Ask about the findings or threat categories...", show_label=False)

                # Session state variables
                current_report_state = gr.State(value={})

    pages = [dashboard_page, analyze_page]
    navigation = [
        (nav_dashboard, "Dashboard"),
        (nav_analyze, "Analyze APK"),
        (dashboard_scan, "Analyze APK")
    ]

    for button, page_name in navigation:
        button.click(
            fn=lambda p=page_name: show_page(p),
            inputs=[],
            outputs=pages + [page_label]
        )

    # Trigger backend analysis and Gemini generation
    def trigger_analysis(apk_path):
        report_text, _, report_json, report_path = run_apk_analysis_ui(apk_path)

        if "Analysis failed" in report_text:
            return (
                f"<div class='glass' style='border-color: #ef4444;'><h3 style='color: #ef4444;'>Scan Failed</h3><p>{report_text}</p></div>",
                "Analysis failed.",
                gr.update(visible=False),
                {},
                [["System notification", "An error occurred during verification. Try another APK file."]]
            )

        resnet_pred = report_json.get("model_results", {}).get("ResNet50", {}).get("predicted_class", "Unknown")
        mobile_pred = report_json.get("model_results", {}).get("MobileNetV3-Large", {}).get("predicted_class", "Unknown")

        final_prediction = resnet_pred if resnet_pred == mobile_pred else f"{resnet_pred} (Primary) / {mobile_pred}"
        color = "#34d399" if "benign" in final_prediction.lower() else "#f87171"

        headline_html = f"""
        <div class="glass" style="border-left: 6px solid {color};">
            <p class="muted" style="margin: 0; font-size: 14px; text-transform: uppercase;">Final Threat Classification</p>
            <h2 style="color: {color}; margin: 8px 0 0 0; font-size: 28px;">{final_prediction.upper()}</h2>
        </div>
        """

        ai_report_content = generate_ai_report(report_json)
        initial_bot_message = ["Hello", f"I have loaded the AI Report for `{report_json.get('filename')}`. Ask me anything about the analysis!"]

        dl_btn_update = gr.update(value=report_path, visible=True) if report_path else gr.update(visible=False)

        return headline_html, ai_report_content, dl_btn_update, report_json, [initial_bot_message]

    # Chatbot response loop
    def chatbot_respond(user_message, history, report_data):
        if not user_message.strip():
            return "", history

        if not history:
            history = []

        if not report_data:
            reply = "Please run an APK scan first so I can analyze the findings."
        elif not GEMINI_AVAILABLE:
            reply = f"I am currently running in fallback mode without an API key, but I see the file analyzed is `{report_data.get('filename')}`."
        else:
            try:
                prompt = (
                    f"Context details of analyzed file: {json.dumps(report_data)}\n"
                    f"Chat history: {history}\n"
                    f"User question: {user_message}"
                )
                response = client.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=prompt
                )
                reply = response.text
            except Exception as e:
                reply = f"Error responding with AI: {e}"

        history.append([user_message, reply])
        return "", history

    analyze_btn.click(
        fn=trigger_analysis,
        inputs=apk_input,
        outputs=[predicted_class_output, scan_report, report_download, current_report_state, chatbot]
    )

    chat_input.submit(
        fn=chatbot_respond,
        inputs=[chat_input, chatbot, current_report_state],
        outputs=[chat_input, chatbot]
    )

# Launch advanced interface with debug enabled to troubleshoot direct iframe loading issues
app.launch(
    share=True,
    debug=True,
    theme=gr.themes.Soft(primary_hue="emerald", neutral_hue="slate"),
    css=CSS
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0e36cd03e511101e48.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
import json
from pathlib import Path
from datetime import datetime, timezone

def run_apk_analysis_ui(apk_path):
    if not apk_path:
        return "Upload an APK first.", None, None, None

    try:
        report, byteplot = analyze_apk(apk_path)
        results = report["model_results"]
        r = results["ResNet50"]
        m = results["MobileNetV3-Large"]

        same = r["predicted_class"] == m["predicted_class"]
        lines = [
            "# APK Analysis Report",
            f"**File:** {report['filename']}",
            f"**ResNet50:** {r['predicted_class']}",
            f"**MobileNetV3-Large:** {m['predicted_class']}",
            f"**Models agree:** {'Yes' if same else 'No'}",
            "",
            "| Class | ResNet50 | MobileNetV3-Large |",
            "|---|---:|---:|"
        ]

        for label in class_names:
            lines.append(
                f"| {label} | {r['scores_percent'][label]:.2f}% "
                f"| {m['scores_percent'][label]:.2f}% |"
            )

        lines += [
            "",
            "## Recommendations",
            "- Treat this as an experimental screening result.",
            "- Verify suspicious samples independently.",
            "- Do not install suspicious APKs on a personal device.",
            "",
            "## Limitations",
            "- Byteplot generation and normalization need verification.",
            "- LightGBM was not run; required features are unavailable.",
            "- This is not a confirmed malware verdict."
        ]

        report["agentic_ai"] = {
            "status": "Fallback summary; AI response not confirmed",
            "models_agree": same
        }
        report["generated_at_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        path = Path("/content/latest_malware_report.json")
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")

        return "\n".join(lines), byteplot, report, str(path)

    except Exception as exc:
        return f"Analysis failed: {exc}", None, None, None

print("Analysis function defined successfully.")

Analysis function defined successfully.


In [20]:
# Cell 8: Verify active Gradio interface
try:
    is_running = app.is_running if 'app' in globals() else False
    print(f"MalwareGuard AI Interactive Webpage Active: {is_running}")
    if is_running:
        print("Navigate to the public link in the previous cell output to upload APKs!")
except Exception as e:
    print(f"Could not check app status: {e}")

MalwareGuard AI Interactive Webpage Active: True
Navigate to the public link in the previous cell output to upload APKs!
